In [1]:
import json
from pathlib import Path
from evalforge.utils import *

dataset_path = Path("data/find_helpful_reviews_data_useful_o1_mini.jsonl")
reviews = load_jsonl(dataset_path)

print(f"Number of reviews: {len(reviews)}")


Number of reviews: 13717


In [2]:
reviews[0]

{'review_key': 'B09MKGM82R_AFPU2ESIDLSPF36S4HVS7BL6C7XQ_1643337186929',
 'review': {'review_key': 'B09MKGM82R_AFPU2ESIDLSPF36S4HVS7BL6C7XQ_1643337186929',
  'rating': 5.0,
  'title': 'SO CUTE!!',
  'text': 'This top is really adorable. The material is like a thin linen texture. I love the color and it fits true to size. I ordered a medium and that is what I usually wear. the ruffle sleeves are so cute and the tie at the neck gives it a nice detail. It would be adorable with jeans or shorts in the summer! It is definitely thin so more for warm weather.',
  'asin': 'B09MKGM82R',
  'parent_asin': 'B09MKFRJNN',
  'user_id': 'AFPU2ESIDLSPF36S4HVS7BL6C7XQ',
  'timestamp': 1643337186929,
  'helpful_vote': 15,
  'verified_purchase': False,
  'requires_product_interaction': False,
  'thinking': "The reviewer comments on the top's material, fit, and details, which can be inferred from the product description itself. The review provides a positive signal that the description aligns with the revie

In [3]:
# GPT3o reviews
# flattened_reviews = {review['review_key']: {k: v for d in (review['review'], review['review_evaluation']) for k, v in d.items()} for review in reviews}

# flattened_reviews[next(iter(flattened_reviews))]
# len(flattened_reviews)

In [4]:
# Flatten o1-mini's review and evaluation data
flattened_reviews = {review['review_key']: {k: v for d in (review['review'], {'o1_mini_review_evaluation': review['review_evaluation']}) for k, v in d.items()} for review in reviews}
for review in flattened_reviews.values():
    o1_mini_evaluation = review.pop('o1_mini_review_evaluation')
    for k, v in o1_mini_evaluation.items():
        review[f'o1_mini_{k}'] = v

flattened_reviews[next(iter(flattened_reviews))]

{'review_key': 'B09MKGM82R_AFPU2ESIDLSPF36S4HVS7BL6C7XQ_1643337186929',
 'rating': 5.0,
 'title': 'SO CUTE!!',
 'text': 'This top is really adorable. The material is like a thin linen texture. I love the color and it fits true to size. I ordered a medium and that is what I usually wear. the ruffle sleeves are so cute and the tie at the neck gives it a nice detail. It would be adorable with jeans or shorts in the summer! It is definitely thin so more for warm weather.',
 'asin': 'B09MKGM82R',
 'parent_asin': 'B09MKFRJNN',
 'user_id': 'AFPU2ESIDLSPF36S4HVS7BL6C7XQ',
 'timestamp': 1643337186929,
 'helpful_vote': 15,
 'verified_purchase': False,
 'requires_product_interaction': False,
 'thinking': "The reviewer comments on the top's material, fit, and details, which can be inferred from the product description itself. The review provides a positive signal that the description aligns with the reviewer's experience regarding aesthetics, fit, and appropriateness for the season. The explicit e

In [5]:
import pandas as pd

df = pd.DataFrame.from_dict(flattened_reviews, orient='index')
useful_count = df['o1_mini_is_useful'].sum()
print(f"Count of is_useful == True: {useful_count}")


Count of is_useful == True: 3899


In [7]:
useful_reviews = df[df['o1_mini_is_useful']].to_dict(orient='index')
useful_reviews_list = [{'review_key': k, **v} for k, v in useful_reviews.items()]

In [8]:
# Write to file

# useful_reviews_path = Path("data/find_helpful_reviews_data_useful.jsonl")
useful_reviews_path = Path("data/find_helpful_reviews_data_useful_o1_mini_useful.jsonl")
with useful_reviews_path.open('w') as f:
    for review in useful_reviews_list:
        f.write(json.dumps(review) + '\n')

In [10]:
useful_reviews_list[0]

{'review_key': 'B09MKGM82R_AFPU2ESIDLSPF36S4HVS7BL6C7XQ_1643337186929',
 'rating': 5.0,
 'title': 'SO CUTE!!',
 'text': 'This top is really adorable. The material is like a thin linen texture. I love the color and it fits true to size. I ordered a medium and that is what I usually wear. the ruffle sleeves are so cute and the tie at the neck gives it a nice detail. It would be adorable with jeans or shorts in the summer! It is definitely thin so more for warm weather.',
 'asin': 'B09MKGM82R',
 'parent_asin': 'B09MKFRJNN',
 'user_id': 'AFPU2ESIDLSPF36S4HVS7BL6C7XQ',
 'timestamp': 1643337186929,
 'helpful_vote': 15,
 'verified_purchase': False,
 'requires_product_interaction': False,
 'thinking': "The reviewer comments on the top's material, fit, and details, which can be inferred from the product description itself. The review provides a positive signal that the description aligns with the reviewer's experience regarding aesthetics, fit, and appropriateness for the season. The explicit e

In [11]:
import ipywidgets as widgets
from IPython.display import display

class ReviewViewer:
    def __init__(self, reviews):
        self.reviews = reviews
        self.index = 0
        self.text_area = widgets.Textarea(value=self.reviews[self.index]['text'], layout=widgets.Layout(width='100%', height='200px'))
        self.requires_interaction = widgets.Label(value=f"Requires Product Interaction: {self.reviews[self.index]['o1_mini_requires_product_interaction']}")
        self.thinking = widgets.Textarea(value=self.reviews[self.index]['o1_mini_thinking'], layout=widgets.Layout(width='100%', height='100px'))
        self.prev_button = widgets.Button(description="Previous")
        self.next_button = widgets.Button(description="Next")
        self.prev_button.on_click(self.prev_review)
        self.next_button.on_click(self.next_review)
        self.update_display()
        display(widgets.VBox([self.text_area, self.requires_interaction, self.thinking, widgets.HBox([self.prev_button, self.next_button])]))

    def update_display(self):
        self.text_area.value = self.reviews[self.index]['text']
        self.requires_interaction.value = f"Requires Product Interaction: {self.reviews[self.index]['o1_mini_requires_product_interaction']}"
        self.thinking.value = self.reviews[self.index]['o1_mini_thinking']

    def prev_review(self, b):
        if self.index > 0:
            self.index -= 1
            self.update_display()

    def next_review(self, b):
        if self.index < len(self.reviews) - 1:
            self.index += 1
            self.update_display()

review_viewer = ReviewViewer(useful_reviews_list)
